# Notebook 07 — Phase-Lock Geometry

**Residue Manifold Learning**

This notebook gives a geometric interpretation of the CGCS threshold using a simple cosine/angle model.

Prior notebooks established:

- Notebook 01: mod30 residue-lane structure exists.
- Notebook 02: constraint sampling improves signal access.
- Notebook 03: NMF compactly recovers lane structure.
- Notebook 04: SAE can dilute capacity into redundant or inactive features.
- Notebook 05: method behavior separates into structural regimes.
- Notebook 06: CGCS scores phase-locked vs diluted representations.

Notebook 07 connects CGCS to a **45° phase-lock geometry** using cosine alignment.


In [ ]:
# NOTE:
# Figures are saved as SVG only.
# Do not save PNG duplicates.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

CGCS_GATE = 24 / 25
THETA_GATE_DEG = 45
COS_45 = np.cos(np.deg2rad(THETA_GATE_DEG))


def save_svg(fig, name):
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

print("CGCS gate:", CGCS_GATE)
print("45° cosine gate:", COS_45)


## 1. Cosine alignment model

We use a minimal geometric model:

```text
cos θ = (A · B) / (||A|| ||B||)
```

Interpretation:

- θ ≈ 0°: representation aligns with residue-manifold structure.
- θ ≤ 45°: phase-locked region.
- θ > 45°: degraded or diluted region.

This is a geometric interpretation of structure fidelity, not a claim about a physical law.


In [ ]:
angles_deg = np.linspace(0, 90, 500)
angles_rad = np.deg2rad(angles_deg)
cos_vals = np.cos(angles_rad)

# Structure proxy for visualization only.
# Squaring keeps the quantity bounded and emphasizes degradation with angle.
structure_quality = cos_vals ** 2

df_geom = pd.DataFrame({
    "angle_deg": angles_deg,
    "cosine": cos_vals,
    "structure_quality_proxy": structure_quality,
    "phase_locked_by_angle": angles_deg <= THETA_GATE_DEG,
})

df_geom.head()


## 2. Figure — Cosine phase-lock curve

This figure marks the 45° boundary where cosine alignment equals `1 / √(1² + 1²)`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_geom["angle_deg"], df_geom["cosine"], linewidth=2, label="cos θ")
ax.axvline(THETA_GATE_DEG, linestyle="--", alpha=0.55, label="45° gate")
ax.axhline(COS_45, linestyle="--", alpha=0.55, label="1 / √(1² + 1²)")

ax.text(THETA_GATE_DEG + 1, COS_45 + 0.025, "45° phase-lock", fontsize=9)

ax.set_xlabel("Angle θ (degrees)")
ax.set_ylabel("cos θ")
ax.set_ylim(0, 1.05)
ax.set_title("Cosine Phase-Lock Threshold")
ax.legend(frameon=False)
fig.tight_layout()
save_svg(fig, "cosine_phase_lock_curve")
plt.show()


## 3. Figure — Angle vs structure quality proxy

The proxy `cos² θ` gives a simple bounded curve for visualizing structure loss as angle increases.

The dashed line at `24/25` marks the CGCS gate from Notebook 06.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_geom["angle_deg"], df_geom["structure_quality_proxy"], linewidth=2, label="cos² θ proxy")
ax.axvline(THETA_GATE_DEG, linestyle="--", alpha=0.55, label="45° gate")
ax.axhline(CGCS_GATE, linestyle="--", alpha=0.55, label="24/25 CGCS gate")

ax.text(1, CGCS_GATE + 0.01, "24/25 threshold", fontsize=9, verticalalignment="bottom")
ax.text(THETA_GATE_DEG + 1, 0.52, "45°", fontsize=9)

ax.set_xlabel("Angle θ (degrees)")
ax.set_ylabel("Structure quality proxy")
ax.set_ylim(0, 1.05)
ax.set_title("Angle vs Structure Quality Proxy")
ax.legend(frameon=False)
fig.tight_layout()
save_svg(fig, "angle_vs_structure_quality")
plt.show()


## 4. Load CGCS scores

Preferred input:

```text
data/cgcs_scores.csv
```

If the file is absent in a fresh Colab runtime, this notebook generates compact fallback scores so the geometry notebook remains runnable.


In [ ]:
path = "data/cgcs_scores.csv"

if os.path.exists(path):
    df_cgcs = pd.read_csv(path)
    print(f"Loaded: {path}")
else:
    print("Missing data/cgcs_scores.csv; generating compact fallback CGCS data.")
    records = []
    for k in range(1, 13):
        coverage = min(k / 8, 1.0)
        lane_mass_ratio = 1.0
        redundancy_penalty = 1 / (1 + max(k - 8, 0))
        dead_feature_penalty = 1.0
        reconstruction_penalty = 1 / (1 + 0.02 / (k + 1))
        cgcs = coverage * lane_mass_ratio * redundancy_penalty * dead_feature_penalty * reconstruction_penalty
        records.append({
            "method": "NMF",
            "capacity": k,
            "topk": np.nan,
            "coverage": coverage,
            "lane_mass_ratio": lane_mass_ratio,
            "redundancy_penalty": redundancy_penalty,
            "dead_feature_penalty": dead_feature_penalty,
            "reconstruction_penalty": reconstruction_penalty,
            "reconstruction_mse": 0.02 / (k + 1),
            "cgcs": cgcs,
        })

    for topk in [1, 2, 4]:
        for cap in [8, 12, 16, 24, 32]:
            coverage = float(np.clip(0.45 + 0.10 * np.log2(cap / 8 + 1) + {1: -0.10, 2: 0.05, 4: 0.00}[topk], 0.25, 0.875))
            lane_mass_ratio = float(np.clip(0.72 + 0.05 * topk - 0.004 * max(cap - 16, 0), 0.55, 0.95))
            dead = int(max(0, cap - int(coverage * 8) - topk * 2))
            redundant = int(max(0, cap * coverage - 8 * coverage))
            mse = float(0.018 / (1 + 0.2 * cap) + 0.002 / topk)
            redundancy_penalty = 1 / (1 + redundant)
            dead_feature_penalty = 1 / (1 + dead)
            reconstruction_penalty = 1 / (1 + mse / 0.02)
            cgcs = coverage * lane_mass_ratio * redundancy_penalty * dead_feature_penalty * reconstruction_penalty
            records.append({
                "method": "SAE",
                "capacity": cap,
                "topk": topk,
                "coverage": coverage,
                "lane_mass_ratio": lane_mass_ratio,
                "redundancy_penalty": redundancy_penalty,
                "dead_feature_penalty": dead_feature_penalty,
                "reconstruction_penalty": reconstruction_penalty,
                "reconstruction_mse": mse,
                "cgcs": cgcs,
            })

    df_cgcs = pd.DataFrame(records)
    df_cgcs["cgcs_gate"] = np.where(df_cgcs["cgcs"] >= CGCS_GATE, "phase-locked", np.where(df_cgcs["cgcs"] >= 0.75, "partial", "diluted"))
    df_cgcs.to_csv(path, index=False)
    print(f"Saved fallback: {path}")

if "cgcs" not in df_cgcs.columns:
    raise ValueError("Expected a 'cgcs' column in CGCS data.")

print(df_cgcs.head())
print("Rows:", len(df_cgcs))


## 5. Map CGCS to implied geometric angle

For visualization, invert CGCS through arccos:

```text
implied_angle = arccos(CGCS)
```

This does **not** claim CGCS is literally cosine. It provides an interpretable angle coordinate for comparing scores to a 45° gate.


In [ ]:
df_cgcs = df_cgcs.copy()
df_cgcs["cgcs_clipped"] = np.clip(df_cgcs["cgcs"], 0, 1)
df_cgcs["implied_angle_deg"] = np.degrees(np.arccos(df_cgcs["cgcs_clipped"]))
df_cgcs["phase_locked_by_angle"] = df_cgcs["implied_angle_deg"] <= THETA_GATE_DEG

df_cgcs[["method", "capacity", "topk", "cgcs", "implied_angle_deg", "phase_locked_by_angle"]].head(12)


## 6. Figure — CGCS vs implied angle

This figure places method runs on a geometric angle coordinate.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df_cgcs.groupby("method"):
    ax.scatter(
        group["implied_angle_deg"],
        group["cgcs"],
        alpha=0.85,
        label=method,
    )

ax.axvline(THETA_GATE_DEG, linestyle="--", alpha=0.55, label="45° gate")
ax.axhline(CGCS_GATE, linestyle="--", alpha=0.55, label="24/25 CGCS gate")

ax.text(THETA_GATE_DEG + 1, 0.08, "45°", fontsize=9)
ax.text(1, CGCS_GATE + 0.01, "24/25 threshold", fontsize=9, verticalalignment="bottom")

ax.set_xlabel("Implied angle from CGCS (degrees)")
ax.set_ylabel("CGCS")
ax.set_xlim(0, 90)
ax.set_ylim(0, 1.05)
ax.set_title("CGCS vs Geometric Angle")
ax.legend(frameon=False)
fig.tight_layout()
save_svg(fig, "cgcs_vs_cosine")
plt.show()


## 7. Figure — Phase-lock region

This figure shades the geometric phase-locked region (`θ ≤ 45°`) against the degraded region (`θ > 45°`).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_geom["angle_deg"], df_geom["structure_quality_proxy"], linewidth=2, label="cos² θ proxy")

ax.fill_between(
    df_geom["angle_deg"],
    df_geom["structure_quality_proxy"],
    where=df_geom["angle_deg"] <= THETA_GATE_DEG,
    alpha=0.22,
    label="phase-locked region",
)

ax.fill_between(
    df_geom["angle_deg"],
    df_geom["structure_quality_proxy"],
    where=df_geom["angle_deg"] > THETA_GATE_DEG,
    alpha=0.18,
    label="degraded region",
)

ax.axvline(THETA_GATE_DEG, linestyle="--", alpha=0.55)
ax.axhline(CGCS_GATE, linestyle="--", alpha=0.55, label="24/25 CGCS gate")

ax.set_xlabel("Angle θ (degrees)")
ax.set_ylabel("Structure quality proxy")
ax.set_ylim(0, 1.05)
ax.set_title("Phase-Lock Geometry")
ax.legend(frameon=False)
fig.tight_layout()
save_svg(fig, "phase_lock_region")
plt.show()


## 8. Save geometry data

Save the analytic curve and the CGCS angle projection for later paper figures or repo docs.


In [ ]:
df_geom.to_csv("data/phase_lock_geometry.csv", index=False)
df_cgcs.to_csv("data/cgcs_phase_lock_angles.csv", index=False)

print("Saved: data/phase_lock_geometry.csv")
print("Saved: data/cgcs_phase_lock_angles.csv")


## 9. Paper claim

> The CGCS threshold admits a simple geometric interpretation: high structural fidelity corresponds to small alignment angle, while degraded representations move toward larger implied angles. The 45° boundary provides an interpretable phase-lock gate for residue-manifold learning.

This is a controlled geometry bridge for this project’s metric and experiments. It should be described as an interpretive model, not a universal physical law.


In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "07_phase_lock_geometry_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)
